# 📘 Annotated edition · 00_template

This is a copy of `notebooks/00_template.ipynb`. Every original cell is unchanged, with its saved output. The cells marked **📘 Explanation** and **✅ Answer** are new.

**What this notebook is:** it isn't an analysis. It's the blank form every new notebook in the project copies. It fixes a structure (CONFIG, then MAP → DO → SEE → CHECK → SUM for each section) so any reader, such as a teammate, the panel or the finance controller, can open any notebook and know where to look.

**Why the saved outputs show errors:** the code cells use placeholders such as `<table>.csv`. The `FileNotFoundError` and `NameError` messages are expected. They show that the template was run once without being filled in. They don't mean the project is broken.

**How to use it:** copy it, rename it `NN_name.ipynb` (NN = run order), replace every `<...>` placeholder, and delete the example comments you don't need.

# <NN>_<name> · Brief <LETTER> · Group <N>

**What this notebook produces:** <one line: the table or number it writes, and where>.

**What it needs:** <files in data/raw or data/clean it reads>.

**Owner:** <name>. **Approach-log rows:** <numbers>.

Every section follows the same five cells: MAP (where this sits and what this section adds), DO (one idea per code cell), SEE (print the output), CHECK (an assert that prints pass or fail without stopping the class), SUM (what we now know, with its denominator).

#### 📘 Explanation: the header cell

The header answers four questions before any code runs:

| Line | Why it exists |
|---|---|
| **Title `<NN>_<name> · Brief · Group`** | The number is the **run order**. Notebook 03 reads what 02 wrote, so running them out of order gives wrong or missing inputs. |
| **What this notebook produces** | Names the output file or number. A reviewer can then check it exists and has the stated row count. |
| **What it needs** | Its inputs. If an input is missing, you know which earlier notebook to run. |
| **Owner / Approach-log rows** | Who answers questions about it, and which rows in `docs/approach-log.md` record the choices made in it. |

**The five-cell rhythm:**
- **MAP**: where this section sits in the pipeline *Sources → Profile → Clean and reconcile → Analyse → Answer*, and what it adds.
- **DO**: one idea per code cell. Small cells make it easy to find which step changed a number.
- **SEE**: print the result. A step that isn't printed can't be checked by eye.
- **CHECK**: an assert-style check that prints PASS or FAIL **without stopping the notebook**. A class demo, or a run that has to finish, can't die halfway on one failed check, but the failure must still be visible.
- **SUM**: one or two sentences on what we now know, **with its denominator and period** (for example "4.2% of 563,380 bookings, H1 2026"). A number without a denominator can't be audited.

✅ **Answer, filled in for this project** (example for notebook 01):
> **01_profile · Brief A · Group N.** *Produces:* `data/clean/profile_files.csv`, `profile_columns.csv`. *Needs:* `data/raw/` (368 files). *Owner:* Rohit. *Approach-log rows:* 1, 3, 6.

## CONFIG

Everything instance-specific lives here, so the notebook can be re-run on another half-year or another city by changing one cell.

#### 📘 Explanation: why a CONFIG cell

Everything that changes between runs (brief letter, folders, the two periods compared, the random seed) sits in **one cell at the top**. If the COO asks for "the same analysis for H2", you edit two dates in one place instead of hunting through 40 cells. It also keeps "magic numbers" out of the analysis code. A date written into a filter halfway down the notebook is the kind of thing that gets forgotten when the period changes.

#### 📘 Explanation: the CONFIG code, line by line

- `CONFIG = {...}`: a plain dictionary, so every later cell reads `CONFIG['raw_dir']` and so on.
  - `'brief'`, `'group'`: labels printed on outputs such as the Excel board number.
  - `'raw_dir'`, `'clean_dir'`: **relative** paths. Raw data is never edited. Every derived file goes to `data/clean`, so you can delete `data/clean` and rebuild it.
  - `'period_this'` / `'period_last'`: the comparison is H1 2026 against H1 2025, the **same half a year apart**, so seasonality cancels out. (Comparing Jan–Jun with Jul–Dec would mix in seasonal swings.)
  - `'seed': 20261019`: fixes random numbers (shuffles, samples) so a re-run gives identical output. A result that changes on every run can't be audited.
- `np.random.seed(...)`: applies the seed to NumPy's legacy global generator. The later notebooks use the newer `np.random.default_rng(seed)`, which is preferred because it's an explicit object you pass around rather than hidden global state.
- `pd.set_option('display.max_columns', 40)`: wide tables print all their columns instead of `...`.

⚠️ **A trap to know about:** the periods are written as dates with no time. Filtering a timestamp with `between('2026-01-01', '2026-06-30')` **excludes everything on 30 June after 00:00**. Notebook 04 shows this costs about 1,000 bookings per half. Prefer a half-open interval: `slot_at >= '2026-01-01'` and `slot_at < '2026-07-01'`.

In [4]:
CONFIG = {
    'brief': '<LETTER>',
    'group': '<N>',
    'raw_dir': 'data/raw',
    'clean_dir': 'data/clean',
    'period_this': ('2026-01-01', '2026-06-30'),   # first half of this year, as released
    'period_last': ('2025-01-01', '2025-06-30'),   # same half of last year
    'seed': 20261019,
}

import pandas as pd
import numpy as np
np.random.seed(CONFIG['seed'])
pd.set_option('display.max_columns', 40)

## 1 · MAP

<Where this notebook sits in the pipeline: Sources → Profile → Clean and reconcile → Analyse → Answer. What this section adds.>

#### 📘 Explanation: MAP

The MAP cell names the pipeline stage and states what this section adds. Example: *"Stage: Clean and reconcile. This section turns 552,492 raw collection rows into one row per booking by removing double scans."* If you can't write that sentence, the section probably doesn't have one clear job yet.

### DO

<What is about to happen, and why, in two sentences.>

#### 📘 Explanation: DO

Two sentences: *what* is about to happen and *why*. The "why" matters most in an interview, where the panel asks "why did you do that?" and the answer should already be written here.

#### 📘 Explanation: the example DO cell

`pd.read_csv(f"{CONFIG['raw_dir']}/<table>.csv")` builds a path from CONFIG with an f-string and reads the file. `print(raw.shape)` prints `(rows, columns)`. The row count is the first number to write down.

**About the saved error:** `[Errno 2] No such file or directory: 'data/raw/<table>.csv'` appears because `<table>` is a placeholder. In a real copy, change it to, for example, `patients/patients.csv.gz`. Note also that the real pack is **gzip-compressed** (`.csv.gz`). pandas decompresses it automatically from the extension.

In [5]:
# one idea per cell
raw = pd.read_csv(f"{CONFIG['raw_dir']}/<table>.csv")
print(raw.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/<table>.csv'

### SEE

#### 📘 Explanation: SEE

Look at the data, not only at its size: `head()`, `dtypes`, `value_counts()`. "Looking" is how you catch things like dates stored as text or ids with trailing spaces before they break a join.

In [6]:
raw.head()

NameError: name 'raw' is not defined

#### 📘 Explanation: `raw.head()` and its error

`head()` shows the first 5 rows. The saved `NameError: name 'raw' is not defined` follows from the previous cell failing: `raw` was never created. This is a useful lesson about notebooks. **One failed cell makes every cell that depends on it fail.** Always fix the *first* error first.

### CHECK

Asserts print pass or fail and do not stop the run. The headline is recomputed after every step so the bridge from raw to final is visible.

#### 📘 Explanation: CHECK

"Asserts print pass or fail and do not stop the run." A plain Python `assert` raises an exception and halts the notebook. Here the check **records** the failure and carries on, so you see *all* failures in one run, and `checks.summary()` lists them at the end.

"The headline is recomputed after every step." After each cleaning step you re-print the headline number (for example total tests). The sequence *raw → after rule 1 → after rule 2 → final* is the **bridge** the finance controller will ask for.

#### 📘 Explanation: the `check` helper, line by line

```python
def check(name, condition, detail=''):
    print(('PASS ' if condition else 'FAIL ') + name + ('  ' + str(detail) if detail != '' else ''))
```
- Takes a human-readable `name`, a boolean `condition`, and an optional `detail` (usually the numbers behind the check).
- Prints `PASS name  detail` or `FAIL name  detail`. Nothing is raised, so execution continues.

The three example checks are the three most important checks in data work:
1. **`rows present`**: the file wasn't empty.
2. **`input = clean + rejected`**: no row disappears silently during cleaning. Every row is either kept or deliberately rejected, and the two add back to the input.
3. **`join did not fan out`**: after a LEFT join the row count must equal the left table's. If it grows, the right table repeats the key and you have just **double-counted**, for example revenue.

The project's real version lives in `src/kalpa/checks.py`. It also stores every result so `checks.summary()` can print a PASS/FAIL table at the end.

In [7]:
def check(name, condition, detail=''):
    print(('PASS ' if condition else 'FAIL ') + name + ('  ' + str(detail) if detail != '' else ''))

# example checks: replace with yours
check('rows present', len(raw) > 0, len(raw))
# after a cleaning step:
# check('input = clean + rejected', len(raw) == len(clean) + len(rejected), (len(raw), len(clean), len(rejected)))
# after a join:
# check('join did not fan out', joined.shape[0] == left.shape[0], (left.shape[0], joined.shape[0]))

NameError: name 'raw' is not defined

### SUM

<What we now know, with its denominator and period. One or two sentences. Copy the number into decisions-log.md if it is a headline.>

#### 📘 Explanation: SUM

Example of a good SUM, using numbers from notebook 03: *"After dropping 1,499 superseded nova records, 563,380 bookings remain (1,303,719 tests), Jan 2025–Jul 2026, six cities."* It has the number, the unit, the denominator and the period. If it's a headline number, copy it into `docs/decisions-log.md` as well.

## 2 · <next section, same five cells>

#### 📘 Explanation
Every further section repeats MAP → DO → SEE → CHECK → SUM. The repetition is deliberate: a reviewer always knows where the check for a step is.

## Break it on purpose

One cell that shows the wrong way and its exact wrong output, so the group can recognise the failure in the viva: the join that doubles revenue, the sum over a text column, the rate without a denominator.

#### 📘 Explanation: "Break it on purpose"

You show the **wrong** way once, with its wrong output, so you'll recognise the symptom later and can explain it in the viva. The three named failures:
- **The join that doubles revenue**: joining invoices to payments where one invoice has several payment rows (retries), then summing invoice amounts. Each invoice is counted once per payment.
- **The sum over a text column**: `amount` stored as `"1,499.00"` (text). Summing strings *concatenates* them, or raises an error.
- **The rate without a denominator**: "no-shows up 300" means nothing until you know it's 300 out of how many bookings.

#### 📘 Explanation: the fan-out example

- `bad = invoices.merge(payments, on='ref', how='left')`: if a `ref` appears twice in `payments`, each matching invoice row is **duplicated**.
- `print(len(invoices), len(bad))`: if the second number is bigger, the join fanned out.
- `validate='one_to_one'`: tells pandas what the relationship *should* be. If it isn't, pandas raises a `MergeError` instead of silently returning a bigger table ("raises instead of lying").

In this project's real data, notebook 03 found **1,925 invoices with more than one captured payment**. That's exactly the case where this fan-out would inflate collected revenue.

In [ ]:
# example: the join that fans out
# bad = invoices.merge(payments, on='ref', how='left')
# print(len(invoices), len(bad))   # if the second number is larger, a payment reference repeats
# good = invoices.merge(payments, on='ref', how='left', validate='one_to_one')   # raises instead of lying

## Output

<What this notebook wrote to data/clean, with row counts, and the headline number if any.>

#### 📘 Explanation: Output
List what was written, with row counts. Example: *"wrote data/clean/collections_clean.csv.gz, 549,598 rows."* A later notebook (or a reviewer) can then confirm it reads the same number of rows.

#### 📘 Explanation: writing the output

`clean.to_csv(..., index=False)` saves the table without pandas' row index (otherwise an unnamed `0,1,2,...` column appears when reading it back). `print('wrote', len(clean), 'rows')` puts the row count in the output so it can be checked against the next notebook's input count.

In [ ]:
# clean.to_csv(f"{CONFIG['clean_dir']}/<table>_clean.csv", index=False)
# print('wrote', len(clean), 'rows')

---
#### ✅ Questions in this notebook, answered

The template's only "questions" are its placeholders. In short, a filled header answers *what does this produce, from what, who owns it, which decisions does it contain*. Every section answers *where are we, what did we do, what did it look like, did it pass, and what do we now know with what denominator*.